# 02 · Справочники: каналы, объекты, структура тегов

**Цель** — понять, что «говорят» идентификаторы: типы датчиков, инженерные подсистемы, иерархия объектов диспетчера и структура тега инженерной системы (локация: коллектор, пикет, номер канала). Эти данные позже станут пространственными признаками для карты рисков.

In [ ]:
import sys
import pathlib

_HERE = pathlib.Path.cwd()
if _HERE.name == "notebooks":
    RESEARCH = _HERE.parent
else:
    RESEARCH = _HERE
if str(RESEARCH) not in sys.path:
    sys.path.insert(0, str(RESEARCH))

import pandas as pd
import data_utils as du

print("dataset:", du.find_dataset_dir())

# 2.1 · Справочник каналов датчиков

11 485 каналов. Ключевые колонки: `тип_инж_системы` (подсистема СМВУ), `тип_датчика`, `тег_инженерной_системы` (локация), `название_датчика` (человекочитаемое, с пикетами).

In [ ]:
ref_ch = du.load_ref_channels()
print("Справочник каналов:", ref_ch.shape)
ref_ch.head()

print("\nТипы датчиков (топ-19):")
print(ref_ch["тип_датчика"].value_counts().to_string())

print("\nИнженерные подсистемы:")
print(ref_ch["тип_инж_системы"].value_counts().to_string())

# 2.2 · Справочник объектов диспетчера

95 объектов в иерархии `родитель → ребёнок`. Вид объекта: `district` (район), `controlHouse` (диспетчерская), `guardObject` (охранный объект). Прямой связи «канал → ид_объект» нет — только через локации/названия.

In [ ]:
ref_obj = du.load_ref_objects()
print("Справочник объектов:", ref_obj.shape)
ref_obj.head(10)

print("\nВиды объектов:")
print(ref_obj["вид_объекта"].value_counts().to_string())

# 2.3 · Структура тега инженерной системы

Пример тега: `15-11.1.131.2.` → разбиение на 5 частей:
- `p1` — индекс коллектора/объекта (847, 914, 257…);
- `p2` — почти всегда `1` (главный контур);
- `p3` — пикет/номер узла (для темп/тепловых — номер ПК);
- `p4` — вспомогательный (255 = «все каналы», 4095 = «зона»);
- `p5` — номер логического канала.

Сохраняем разобранные теги в артефакт `channel_tag_parsed.csv` — он пригодится для карты объектов.

In [ ]:
out_csv = du.find_dataset_dir() / "channel_tag_parsed.csv"
parsed = ref_ch["тег_инженерной_системы"].apply(du.parse_tag).apply(pd.Series)
parsed.insert(0, "ид_канала_данных", ref_ch["ид_канала_данных"].values)
parsed["название_датчика"] = ref_ch["название_датчика"].values
parsed.to_csv(out_csv, index=False, encoding="utf-8-sig")

print("Топ значений p1 (коллекторы):")
print(parsed["p1"].value_counts().head(15).to_string())
print("\nРаспределение p5 (номер канала в точке):")
print(parsed["p5"].value_counts().head(10).to_string())
parsed.head(8)

# 2.4 · Комментарии

- Типы датчиков покрывают все 4 направления ТЗ: **отказ датчика** (дым/температура/газ/КД), **пожарный риск** (дым/тепловые), **несанкционированный доступ** (движение/КД Дверь/КД АВ), **износ инфраструктуры** (насосы/вентиляторы/ИБП).
- `p1` — надёжный «коллекторный» признак (~20 коллекторов), `p3` — пикет; из них можно собрать пространственные признаки для карты объектов.
- Названия датчиков содержат ПК («ТД ПК28») — при желании парсим в отдельный признак `pk`.